# Two-qubit gate with resonator workflow

This notebook follows the current `Two qubit gate with resonator work flow.md` in one linear flow.

Fixed choices for this version:

- symmetric DQDs
- finite preparation detuning `epsilon_1 = epsilon_2 = 80 GHz`
- DQD1-only lab-frame drive during the preparation stage
- `omega_drive` set by the exact finite-detuning spin-basis qubit resonance and `Omega_drive = 2 pi * 100 MHz`
- explicit linear ramps into and out of the `epsilon = 0` gate point
- readout reported after the return ramp to finite detuning in the spin eigenbasis

Notebook flow:

1. define parameters and build the symmetric DQD-resonator model
2. calibrate the finite-detuning single-qubit preparation
3. estimate the resonator gate time
4. propagate the full workflow sequence for the main prep point
5. run a Bell-state scan using the same prep model
6. inspect the final spin-basis population, charge-traced spin readout, and RR-conditioned spin readout states
7. record the finite-detuning spin-only diagonalization used to move this workflow into a spin-diagonal basis without diagonalizing charge


In [ ]:
from itertools import product
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import qutip as qt
from qutip import basis, qeye, tensor

repo_root = Path.cwd().resolve()
for candidate in [repo_root, *repo_root.parents]:
    if (candidate / "python").exists() and (candidate / "pyproject.toml").exists():
        repo_root = candidate
        break

python_root = repo_root / "python"
if str(python_root) not in sys.path:
    sys.path.insert(0, str(python_root))

from helpers import DQDsystem
from helpers import RAD_PER_NS_PER_MICROELECTRONVOLT, RAD_PER_US_PER_MICROELECTRONVOLT, TWO_PI, single_dqd_qubit_splitting

plt.rcParams["figure.figsize"] = (8, 4)
plt.rcParams["axes.grid"] = True


In [ ]:
def ghz_to_uev(ghz):
    """Convert a frequency given in GHz into the model detuning unit in ueV."""
    return TWO_PI * ghz * 1_000.0 / RAD_PER_US_PER_MICROELECTRONVOLT


def lab_frame_product_state(photon_max, spin1="up", spin2="down"):
    spin_index = {"up": 0, "down": 1}
    psi1 = basis([2, 2], [spin_index[spin1], 0])
    psi2 = basis([2, 2], [spin_index[spin2], 0])
    return tensor(basis(photon_max, 0), psi1, psi2)


def spin_diagonalization_data(dqd):
    theta = np.arctan2(dqd.bx, dqd.Bz)
    spin_field = np.hypot(dqd.Bz, dqd.bx)
    return {
        "theta": float(theta),
        "spin_field": float(spin_field),
        "cos": float(np.cos(theta)),
        "sin": float(np.sin(theta)),
    }


def single_spin_diagonalization_unitary(dqd):
    """Rotate spin only: U H U.dag() diagonalizes Bz*sigma_z + bx*sigma_x*tau_z."""
    theta = spin_diagonalization_data(dqd)["theta"]
    return (0.5j * theta * dqd.sy * dqd.tz).expm()


def full_spin_diagonalization_unitary(dqd):
    U_spin = single_spin_diagonalization_unitary(dqd)
    return tensor(qeye(dqd.photon_max), U_spin, U_spin)


def spin_diagonalized_single_hamiltonian(dqd, epsilon_uev):
    data = spin_diagonalization_data(dqd)
    return (
        dqd.tc * (data["cos"] * dqd.tx - data["sin"] * dqd.sy * dqd.ty)
        + 0.5 * epsilon_uev * dqd.tz
        + 0.5 * data["spin_field"] * dqd.sz
    )


def spin_basis_qubit_resonance_rad_per_us(dqd, epsilon_uev):
    """Exact finite-detuning qubit transition in the spin-diagonal charge basis."""
    energies = np.sort(spin_diagonalized_single_hamiltonian(dqd, epsilon_uev).eigenenergies())
    return float((energies[1] - energies[0]) * RAD_PER_US_PER_MICROELECTRONVOLT)


def transform_full_operator_to_spin_basis(op, dqd):
    U_full = full_spin_diagonalization_unitary(dqd)
    return (U_full * op * U_full.dag()).tidyup(atol=1e-10)


def transform_states_to_spin_basis(states, dqd):
    U_full = full_spin_diagonalization_unitary(dqd)
    return [U_full * state for state in states]


def transform_states_from_spin_basis(states, dqd):
    U_full_dag = full_spin_diagonalization_unitary(dqd).dag()
    return [U_full_dag * state for state in states]


def spin_basis_spin_populations(states, dqd, spin_index):
    return spin_populations(transform_states_to_spin_basis(states, dqd), spin_index)


def spin_basis_two_spin_density_matrix(state, dqd):
    U_full = full_spin_diagonalization_unitary(dqd)
    return qt.ket2dm(U_full * state).ptrace([1, 3])


def spin_basis_rr_spin_density_matrix(state, dqd):
    """Two-spin density matrix conditioned on charge1=R and charge2=R in the spin basis."""
    psi_spin_basis = full_spin_diagonalization_unitary(dqd) * state
    rho_spin_basis = qt.ket2dm(psi_spin_basis)
    proj_r = basis(2, 0) * basis(2, 0).dag()
    proj_rr = tensor(qeye(dqd.photon_max), qeye(2), proj_r, qeye(2), proj_r)
    rho_rr_full = proj_rr * rho_spin_basis * proj_rr
    rr_weight = float(rho_rr_full.tr().real)
    rho_rr_spin = rho_rr_full.ptrace([1, 3])
    if rr_weight > 1e-15:
        rho_rr_spin = rho_rr_spin / rr_weight
    return rho_rr_spin, rr_weight


def spin_basis_rr_spin_populations(state, dqd):
    rho_rr_spin, rr_weight = spin_basis_rr_spin_density_matrix(state, dqd)
    labels = ["++", "+-", "-+", "--"]
    kets = [
        tensor(basis(2, 0), basis(2, 0)),
        tensor(basis(2, 0), basis(2, 1)),
        tensor(basis(2, 1), basis(2, 0)),
        tensor(basis(2, 1), basis(2, 1)),
    ]
    populations = [float(qt.expect(ket * ket.dag(), rho_rr_spin).real) for ket in kets]
    return labels, populations, rr_weight, rho_rr_spin


def spin_basis_two_spin_populations(states, dqd):
    return bare_two_spin_populations(transform_states_to_spin_basis(states, dqd))


def spin_populations(states, spin_index):
    proj_up = basis(2, 0) * basis(2, 0).dag()
    proj_down = basis(2, 1) * basis(2, 1).dag()
    rho_spin = [qt.ket2dm(state).ptrace(spin_index) for state in states]
    p_up = np.array([qt.expect(proj_up, rho) for rho in rho_spin], dtype=float)
    p_down = np.array([qt.expect(proj_down, rho) for rho in rho_spin], dtype=float)
    return p_up, p_down


def first_half_population_crossing(p_up):
    balance = p_up - 0.5
    crossings = np.where(np.sign(balance[:-1]) != np.sign(balance[1:]))[0]
    if len(crossings):
        return int(crossings[0] + 1)
    return int(np.argmin(np.abs(balance)))


def bare_two_spin_populations(states):
    rho_spin_pair = [qt.ket2dm(state).ptrace([1, 3]) for state in states]
    labels = {
        "uu": tensor(basis(2, 0), basis(2, 0)),
        "ud": tensor(basis(2, 0), basis(2, 1)),
        "du": tensor(basis(2, 1), basis(2, 0)),
        "dd": tensor(basis(2, 1), basis(2, 1)),
    }
    return {
        key: np.array([qt.expect(ket * ket.dag(), rho) for rho in rho_spin_pair], dtype=float)
        for key, ket in labels.items()
    }


def charge_right_probability(states, charge_index):
    proj_right = basis(2, 0) * basis(2, 0).dag()
    rho_charge = [qt.ket2dm(state).ptrace(charge_index) for state in states]
    return np.array([qt.expect(proj_right, rho) for rho in rho_charge], dtype=float)


def evolve_under_constant_hamiltonian(H, psi0, tlist):
    evals, evecs = H.eigenstates()
    basis_matrix = np.column_stack([vec.full().ravel() for vec in evecs])
    psi0_vec = psi0.full().ravel()
    coeffs = basis_matrix.conj().T @ psi0_vec
    states = []
    for t in tlist:
        phases = np.exp(-1j * evals * t)
        state_vec = basis_matrix @ (coeffs * phases)
        states.append(qt.Qobj(state_vec[:, None], dims=psi0.dims))
    return states


def bare_two_spin_density_matrix(state):
    return qt.ket2dm(state).ptrace([1, 3])


def mixed_state_concurrence(rho):
    sigma_y = np.array([[0.0, -1.0j], [1.0j, 0.0]], dtype=complex)
    yy = np.kron(sigma_y, sigma_y)
    rho_mat = rho.full()
    R = rho_mat @ yy @ rho_mat.conj() @ yy
    eigvals = np.linalg.eigvals(R)
    lambdas = np.sort(np.sqrt(np.maximum(np.real_if_close(eigvals), 0.0)))[::-1]
    return float(max(0.0, lambdas[0] - lambdas[1] - lambdas[2] - lambdas[3]))


def project_energy_qubit_amplitudes(state, dqd):
    H_single = dqd.tc * dqd.tx + dqd.Bz / 2 * dqd.sz + dqd.bx / 2 * dqd.sx * dqd.tz
    _, evecs = H_single.eigenstates(sort="low")
    g = evecs[0]
    e = evecs[1]
    qbasis = [
        tensor(basis(dqd.photon_max, 0), g, g),
        tensor(basis(dqd.photon_max, 0), g, e),
        tensor(basis(dqd.photon_max, 0), e, g),
        tensor(basis(dqd.photon_max, 0), e, e),
    ]
    amps = np.array([ket.overlap(state) for ket in qbasis], dtype=complex)
    weight = float(np.vdot(amps, amps).real)
    if weight <= 1e-15:
        return amps, weight, amps
    return amps, weight, amps / np.sqrt(weight)


def pure_state_concurrence(amps):
    sigma_y = np.array([[0.0, -1.0j], [1.0j, 0.0]], dtype=complex)
    yy = np.kron(sigma_y, sigma_y)
    return float(abs(amps.T @ yy @ amps))


def ramp_detuning(dqd, psi0, epsilon_start_uev, epsilon_stop_uev, ramp_time_us, ramp_samples, options):
    def ramp_coeff(t, epsilon_start_uev, epsilon_stop_uev, ramp_time_us):
        if ramp_time_us <= 0.0:
            return epsilon_stop_uev
        s = min(max(t / ramp_time_us, 0.0), 1.0)
        return epsilon_start_uev + (epsilon_stop_uev - epsilon_start_uev) * s

    ramp_tlist_us = np.linspace(0.0, ramp_time_us, ramp_samples)
    ramp_result = qt.mesolve(
        [dqd.H_static, [dqd.H_eps1_op, ramp_coeff], [dqd.H_eps2_op, ramp_coeff]],
        psi0,
        ramp_tlist_us,
        c_ops=[],
        e_ops=[],
        args={
            "epsilon_start_uev": epsilon_start_uev,
            "epsilon_stop_uev": epsilon_stop_uev,
            "ramp_time_us": ramp_time_us,
        },
        options=options,
    )
    return ramp_tlist_us, ramp_result.states


def ramp_detuning_to_zero(dqd, psi0, epsilon_start_uev, ramp_time_us, ramp_samples, options):
    return ramp_detuning(dqd, psi0, epsilon_start_uev, 0.0, ramp_time_us, ramp_samples, options)


def vacuum_spin_charge_basis_populations(states, dqd):
    spin_basis_states = transform_states_to_spin_basis(states, dqd)
    spin_charge_labels_single = ["R+", "L+", "R-", "L-"]
    labels = []
    basis_states = []
    for i, j in product(range(4), repeat=2):
        labels.append(f"{spin_charge_labels_single[i]},{spin_charge_labels_single[j]}")
        basis_states.append(
            tensor(
                basis(dqd.photon_max, 0),
                basis([2, 2], [i // 2, i % 2]),
                basis([2, 2], [j // 2, j % 2]),
            )
        )
    populations = np.array([
        [float(abs(ket.overlap(state)) ** 2) for ket in basis_states]
        for state in spin_basis_states
    ])
    return labels, populations


def phase_optimized_odd_bell_overlap(rho):
    """Max overlap with (|+-> + exp(i phi)|-+>)/sqrt(2)."""
    mat = rho.full()
    return float((0.5 * (mat[1, 1].real + mat[2, 2].real) + abs(mat[1, 2])).real)


def phase_optimized_odd_bell_phase(rho):
    """Phase phi for (|+-> + exp(i phi)|-+>)/sqrt(2)."""
    return float(-np.angle(rho.full()[1, 2]))


def odd_bell_target_with_phase(phi):
    return np.array([0.0, 1.0, np.exp(1j * phi), 0.0], dtype=complex) / np.sqrt(2.0)


def bell_metrics(state, dqd, bell_target):
    spin_rho = spin_basis_two_spin_density_matrix(state, dqd)
    rr_spin_rho, rr_spin_weight = spin_basis_rr_spin_density_matrix(state, dqd)
    spin_bell_ket = qt.Qobj(bell_target[:, None], dims=[[2, 2], [1, 1]])
    spin_bell_overlap = float(qt.expect(spin_bell_ket * spin_bell_ket.dag(), spin_rho).real)
    rr_spin_bell_overlap = float(qt.expect(spin_bell_ket * spin_bell_ket.dag(), rr_spin_rho).real)
    spin_phase_optimized_bell_overlap = phase_optimized_odd_bell_overlap(spin_rho)
    rr_spin_phase_optimized_bell_overlap = phase_optimized_odd_bell_overlap(rr_spin_rho)
    spin_phase_optimized_bell_phase = phase_optimized_odd_bell_phase(spin_rho)
    rr_spin_phase_optimized_bell_phase = phase_optimized_odd_bell_phase(rr_spin_rho)
    spin_concurrence = mixed_state_concurrence(spin_rho)
    rr_spin_concurrence = mixed_state_concurrence(rr_spin_rho) if rr_spin_weight > 1e-15 else 0.0

    _, qubit_weight, qubit_amps = project_energy_qubit_amplitudes(state, dqd)
    if qubit_weight <= 1e-15:
        projected_bell_overlap = 0.0
        projected_concurrence = 0.0
        projected_populations = np.zeros(4)
    else:
        projected_bell_overlap = float(abs(np.vdot(bell_target, qubit_amps)) ** 2)
        projected_concurrence = pure_state_concurrence(qubit_amps)
        projected_populations = np.abs(qubit_amps) ** 2

    return {
        "spin_bell_overlap": spin_bell_overlap,
        "spin_phase_optimized_bell_overlap": spin_phase_optimized_bell_overlap,
        "spin_phase_optimized_bell_phase_rad": spin_phase_optimized_bell_phase,
        "spin_concurrence": spin_concurrence,
        "rr_spin_weight": rr_spin_weight,
        "rr_spin_bell_overlap": rr_spin_bell_overlap,
        "rr_spin_phase_optimized_bell_overlap": rr_spin_phase_optimized_bell_overlap,
        "rr_spin_phase_optimized_bell_phase_rad": rr_spin_phase_optimized_bell_phase,
        "rr_spin_weighted_bell_overlap": rr_spin_weight * rr_spin_bell_overlap,
        "rr_spin_weighted_phase_optimized_bell_overlap": rr_spin_weight * rr_spin_phase_optimized_bell_overlap,
        "rr_spin_concurrence": rr_spin_concurrence,
        "projected_bell_overlap": projected_bell_overlap,
        "projected_concurrence": projected_concurrence,
        "qubit_subspace_weight": float(qubit_weight),
        "full_state_bell_weight": float(qubit_weight * projected_bell_overlap),
        "projected_populations": projected_populations,
    }


def bell_candidate_score(candidate):
    return (
        candidate["rr_spin_weighted_phase_optimized_bell_overlap"],
        candidate["rr_spin_phase_optimized_bell_overlap"],
        candidate["rr_spin_concurrence"],
    )


def best_bell_candidate(states, gate_tlist_us, prep_idx, prep_time_ns, dqd, bell_target):
    best = None
    for gate_idx, state in enumerate(states):
        candidate = {
            "prep_idx": int(prep_idx),
            "gate_idx": int(gate_idx),
            "prep_time_ns": float(prep_time_ns),
            "gate_hold_us": float(gate_tlist_us[int(gate_idx)]),
            **bell_metrics(state, dqd, bell_target),
        }
        if best is None or bell_candidate_score(candidate) > bell_candidate_score(best):
            best = candidate
    return best


def plot_population_triptych(time_axis, spin_pair, two_spin_pops, charge_pair, xlabel, titles):
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))

    axes[0].plot(time_axis, spin_pair[0], label="DQD1 spin up")
    axes[0].plot(time_axis, spin_pair[1], label="DQD1 spin down")
    axes[0].set_xlabel(xlabel)
    axes[0].set_ylabel("population")
    axes[0].set_title(titles[0])
    axes[0].legend()

    for label, values in two_spin_pops.items():
        axes[1].plot(time_axis, values, label=label)
    axes[1].set_xlabel(xlabel)
    axes[1].set_ylabel("population")
    axes[1].set_title(titles[1])
    axes[1].legend()

    axes[2].plot(time_axis, charge_pair[0], label="DQD1 charge-right")
    axes[2].plot(time_axis, charge_pair[1], label="DQD2 charge-right")
    axes[2].set_xlabel(xlabel)
    axes[2].set_ylabel("probability")
    axes[2].set_title(titles[2])
    axes[2].legend()

    plt.tight_layout()


def plot_population_basis_bar(labels, populations, title):
    fig, ax = plt.subplots(figsize=(12, 4))
    ax.bar(labels, populations)
    ax.set_ylim(0.0, 1.0)
    ax.set_ylabel("population")
    ax.set_xlabel("spin-charge basis state")
    ax.set_title(title)
    ax.tick_params(axis="x", rotation=45)
    plt.tight_layout()


## 1. Parameters and model

Build one symmetric two-DQD plus resonator model, convert the workflow detuning from GHz to the Hamiltonian unit in ueV, and report the main derived scales used later in the notebook.


In [ ]:
# Symmetric DQD parameters inherited from DQDlabframe.ipynb
tc = ghz_to_uev(10.0)
bx = 3
Bz = 35.0
wc = TWO_PI * 5.5e3
gc = TWO_PI * 50.0
photon_max = 10

# Workflow-specific inputs fixed in the markdown discussion
epsilon_prepare_ghz = 80.0
epsilon_prepare_uev = ghz_to_uev(epsilon_prepare_ghz)
Omega_drive = TWO_PI * 100.0
charge_period_ns = TWO_PI / (2.0 * tc * RAD_PER_NS_PER_MICROELECTRONVOLT)
epsilon_ramp_cycles = 20.0
epsilon_ramp_ns = epsilon_ramp_cycles * charge_period_ns
epsilon_ramp_us = epsilon_ramp_ns / 1000.0
ramp_points_per_charge_period = 80
ramp_samples = int(np.ceil(epsilon_ramp_cycles * ramp_points_per_charge_period)) + 1

dqd = DQDsystem(
    tc=tc,
    bx=bx,
    Bz=Bz,
    Vac0=0.0,
    wc=wc,
    gc=gc,
    photon_max=photon_max,
    epsilon_idle=0.0,
)

prep_resonance_rad_per_us = spin_basis_qubit_resonance_rad_per_us(dqd, epsilon_prepare_uev)
prep_resonance_angular_GHz = prep_resonance_rad_per_us / 1_000.0
prep_resonance_GHz = prep_resonance_rad_per_us / (TWO_PI * 1_000.0)
prep_qubit_split_angular_GHz = single_dqd_qubit_splitting(dqd, epsilon_prepare_uev)
omega_drive = prep_resonance_rad_per_us
gate_time_us = dqd.iSWAP_gate_time()
r_sigma, r_tau = dqd.dispersive_ratios()
gate_estimate = {
    "Esigma_uev": float(dqd.Esigma),
    "phi_bar_rad": float(dqd.phi_bar),
    "d_sigma_rad_per_us": float(dqd.d_sigma),
    "g_sigma_rad_per_us": float(dqd.g_sigma),
    "d_sigma_MHz": float(dqd.d_sigma / TWO_PI),
    "g_sigma_MHz": float(dqd.g_sigma / TWO_PI),
    "gate_time_us": float(gate_time_us),
}

print({
    "epsilon_prepare_uev": epsilon_prepare_uev,
    "omega_drive_rad_per_us": omega_drive,
    "Omega_drive_rad_per_us": Omega_drive,
    "prep_resonance_angular_GHz": prep_resonance_angular_GHz,
    "prep_resonance_GHz": prep_resonance_GHz,
    "old_Bz_angular_GHz": Bz * RAD_PER_US_PER_MICROELECTRONVOLT / 1_000.0,
    "qubit_split_check_angular_GHz": prep_qubit_split_angular_GHz,
    "charge_period_ns": charge_period_ns,
    "epsilon_ramp_ns": epsilon_ramp_ns,
    "ramp_samples": ramp_samples,
    "gate_time_us": gate_time_us,
    "|g_sigma/d_sigma|": r_sigma,
    "|g_tau/d_tau|": r_tau,
})

gate_estimate


## 2. Spin-only diagonalization at finite detuning

For one DQD in the lab charge basis,

$$H = t_c\tau_x + \frac{\epsilon}{2}\tau_z + \frac{B_z}{2}\sigma_z + \frac{b_x}{2}\sigma_x\tau_z.$$

The finite detuning term already commutes with the charge-conditioned spin rotation

$$U_s = \exp\left(i\frac{\theta}{2}\sigma_y\tau_z\right),\qquad \theta = \arctan(b_x/B_z),\qquad B=\sqrt{B_z^2+b_x^2}.$$

Using the notebook convention $H' = U_s H U_s^\dagger$, this diagonalizes only the local spin field:

$$H' = t_c\left(\cos\theta\,\tau_x - \sin\theta\,\sigma_y\tau_y\right) + \frac{\epsilon}{2}\tau_z + \frac{B}{2}\sigma_z.$$

The charge basis is not diagonalized. Consequently, the detuning control and a true EDSR/cavity dipole proportional to $\tau_z$ are unchanged by this spin-only basis change, while a direct spin drive transforms as

$$\sigma_x \rightarrow \cos\theta\,\sigma_x + \sin\theta\,\sigma_z\tau_z.$$

For two DQDs plus the resonator, use $U_{full}=I_{cavity}\otimes U_s^{(1)}\otimes U_s^{(2)}$ and transform every Hamiltonian term as $O' = U_{full} O U_{full}^\dagger$. The EDSR carrier is the exact finite-detuning qubit gap $\omega_{01}=(E_1-E_0)/\hbar$ in this basis, not the bare $B_z/\hbar$ value.


In [ ]:
spin_basis = spin_diagonalization_data(dqd)
U_spin = single_spin_diagonalization_unitary(dqd)

H_single_lab_prepare = (
    dqd.tc * dqd.tx
    + 0.5 * epsilon_prepare_uev * dqd.tz
    + 0.5 * dqd.Bz * dqd.sz
    + 0.5 * dqd.bx * dqd.sx * dqd.tz
)
H_single_spin_basis = (U_spin * H_single_lab_prepare * U_spin.dag()).tidyup(atol=1e-10)
H_single_formula = spin_diagonalized_single_hamiltonian(dqd, epsilon_prepare_uev)

tx_spin_basis = (U_spin * dqd.tx * U_spin.dag()).tidyup(atol=1e-10)
tx_formula = spin_basis["cos"] * dqd.tx - spin_basis["sin"] * dqd.sy * dqd.ty
tz_spin_basis = (U_spin * dqd.tz * U_spin.dag()).tidyup(atol=1e-10)
sx_spin_basis = (U_spin * dqd.sx * U_spin.dag()).tidyup(atol=1e-10)
sx_formula = spin_basis["cos"] * dqd.sx + spin_basis["sin"] * dqd.sz * dqd.tz

spin_diagonal_checks = {
    "||UHUdag - H_formula||": float((H_single_spin_basis - H_single_formula).norm()),
    "||U tau_x Udag - tau_x_formula||": float((tx_spin_basis - tx_formula).norm()),
    "||U tau_z Udag - tau_z||": float((tz_spin_basis - dqd.tz).norm()),
    "||U sigma_x Udag - sigma_x_formula||": float((sx_spin_basis - sx_formula).norm()),
    "resonance_matches_exact_split_angular_GHz": float(prep_resonance_angular_GHz - prep_qubit_split_angular_GHz),
}

spin_diagonal_summary = {
    "theta_rad": spin_basis["theta"],
    "theta_deg": np.degrees(spin_basis["theta"]),
    "spin_field_uev": spin_basis["spin_field"],
    "epsilon_prepare_uev": epsilon_prepare_uev,
    "prep_resonance_rad_per_us": prep_resonance_rad_per_us,
    "prep_resonance_GHz": prep_resonance_GHz,
    "static_single_DQD": "tc*(cos(theta)*tau_x - sin(theta)*sigma_y*tau_y) + epsilon/2*tau_z + B/2*sigma_z",
    "tau_z_controls": "unchanged: detuning, EDSR dipole, and cavity dipole all remain proportional to tau_z",
    "direct_spin_x_drive": "cos(theta)*sigma_x + sin(theta)*sigma_z*tau_z",
}

print(spin_diagonal_summary)
spin_diagonal_checks


## 3. Calibrate the finite-detuning preparation pulse

The workflow uses a finite-detuning DQD1-only EDSR pulse in the spin-diagonal basis, driven at the exact finite-detuning qubit resonance. This section chooses the shortest pulse that makes the reduced DQD1 spin-basis populations as close as possible to 50/50 in the full driven evolution.


In [ ]:
psi0_lab = lab_frame_product_state(photon_max, spin1="down", spin2="down")
U_full_spin = full_spin_diagonalization_unitary(dqd)
psi0_spin_basis = U_full_spin * psi0_lab
H_drive1_lab = tensor(qeye(photon_max), dqd.sx1)
H_prep_static_lab = dqd.H_static + epsilon_prepare_uev * dqd.H_eps1_op + epsilon_prepare_uev * dqd.H_eps2_op
H_drive1 = transform_full_operator_to_spin_basis(H_drive1_lab, dqd)
H_prep_static = transform_full_operator_to_spin_basis(H_prep_static_lab, dqd)

def drive_coeff(t, Omega_drive, omega_drive):
    return Omega_drive * np.cos(omega_drive * t)

prep_scan_ns = np.linspace(0.0, 10.0, 10001)
prep_scan_us = prep_scan_ns / 1000.0
prep_args = {"Omega_drive": Omega_drive, "omega_drive": omega_drive}
prep_opts = {"nsteps": 200000, "atol": 1e-9, "rtol": 1e-9, "store_states": True}

res_prep = qt.mesolve(
    [H_prep_static, [H_drive1, drive_coeff]],
    psi0_spin_basis,
    prep_scan_us,
    c_ops=[],
    e_ops=[],
    args=prep_args,
    options=prep_opts,
)

p_up_prep, p_down_prep = spin_populations(res_prep.states, spin_index=1)
prep_idx = first_half_population_crossing(p_up_prep)
prep_time_us = float(prep_scan_us[prep_idx])
prep_time_ns = 1_000.0 * prep_time_us
psi_after_prep_spin_basis = res_prep.states[prep_idx]
psi_after_prep = U_full_spin.dag() * psi_after_prep_spin_basis

print({
    "prep_time_ns": prep_time_ns,
    "omega_drive_rad_per_us": omega_drive,
    "omega_drive_GHz": prep_resonance_GHz,
    "p_up_at_prep_time": p_up_prep[prep_idx],
    "p_down_at_prep_time": p_down_prep[prep_idx],
})

plt.plot(prep_scan_ns, p_up_prep, label="DQD1 spin up")
plt.plot(prep_scan_ns, p_down_prep, label="DQD1 spin down")
plt.axvline(prep_time_ns, color="black", linestyle="--", label="chosen prep time")
plt.xlabel("prep time (ns)")
plt.ylabel("population")
plt.title("Finite-detuning DQD1 prep calibration")
plt.legend();


## 4. Run the main workflow sequence

Starting from the calibrated prep point, run the whole sequence in order: finite-detuning prep, ramp to `epsilon = 0`, gate evolution, ramp back to finite detuning, and readout. The spin summaries and 16-state population plots are evaluated in the spin eigenbasis, with charge left explicit.


In [ ]:
ramp_tlist_us, ramp_states = ramp_detuning_to_zero(
    dqd,
    psi_after_prep,
    epsilon_prepare_uev,
    epsilon_ramp_us,
    ramp_samples,
    prep_opts,
)
psi_after_ramp = ramp_states[-1]

gate_samples = 5001
gate_tlist_us = np.linspace(0.0, gate_time_us, gate_samples)
gate_states = evolve_under_constant_hamiltonian(dqd.H_static, psi_after_ramp, gate_tlist_us)

psi_after_gate = gate_states[-1]
p_up_gate, p_down_gate = spin_basis_spin_populations(gate_states, dqd, spin_index=1)
two_spin_pops = spin_basis_two_spin_populations(gate_states, dqd)
charge_right_1 = charge_right_probability(gate_states, charge_index=2)
charge_right_2 = charge_right_probability(gate_states, charge_index=4)

readout_ramp_tlist_us, readout_ramp_states = ramp_detuning(
    dqd,
    psi_after_gate,
    0.0,
    epsilon_prepare_uev,
    epsilon_ramp_us,
    ramp_samples,
    prep_opts,
)
psi_after_readout = readout_ramp_states[-1]
p_up_readout, p_down_readout = spin_basis_spin_populations(readout_ramp_states, dqd, spin_index=1)
readout_two_spin_pops = spin_basis_two_spin_populations(readout_ramp_states, dqd)
readout_charge_right_1 = charge_right_probability(readout_ramp_states, charge_index=2)
readout_charge_right_2 = charge_right_probability(readout_ramp_states, charge_index=4)

sequence_times_ns = np.array([
    0.0,
    prep_time_ns,
    prep_time_ns + epsilon_ramp_ns,
    prep_time_ns + epsilon_ramp_ns + 1_000.0 * gate_time_us,
    prep_time_ns + 2.0 * epsilon_ramp_ns + 1_000.0 * gate_time_us,
])
eps_profile_ghz = np.array([epsilon_prepare_ghz, epsilon_prepare_ghz, 0.0, 0.0, epsilon_prepare_ghz])
drive_profile_mhz = np.array([Omega_drive / TWO_PI, Omega_drive / TWO_PI, 0.0, 0.0, 0.0])

fig, ax = plt.subplots(2, 1, figsize=(8, 5), sharex=True)
ax[0].plot(sequence_times_ns, eps_profile_ghz)
ax[0].set_ylabel("detuning (GHz)")
ax[0].set_title("Workflow pulse schedule")
ax[1].plot(sequence_times_ns, drive_profile_mhz)
ax[1].set_ylabel(r"$\Omega_{drive} / 2\pi$ (MHz)")
ax[1].set_xlabel("sequence time (ns)")
plt.tight_layout()

plot_population_triptych(
    gate_tlist_us,
    (p_up_gate, p_down_gate),
    two_spin_pops,
    (charge_right_1, charge_right_2),
    "gate time (us)",
    (
        "DQD1 spin-basis populations during gate",
        "Spin-basis two-spin populations during gate",
        "Charge-right occupancy during gate",
    ),
)

plot_population_triptych(
    1_000.0 * readout_ramp_tlist_us,
    (p_up_readout, p_down_readout),
    readout_two_spin_pops,
    (readout_charge_right_1, readout_charge_right_2),
    "readout ramp time (ns)",
    (
        "DQD1 spin-basis populations during ramp back",
        "Spin-basis two-spin populations during ramp back",
        "Charge-right occupancy during ramp back",
    ),
)

workflow_basis_labels, workflow_basis_populations = vacuum_spin_charge_basis_populations([psi_after_readout], dqd)
plot_population_basis_bar(
    workflow_basis_labels,
    workflow_basis_populations[0],
    "16 spin-charge-basis populations after the whole sequence",
)

readout_summary = {
    "prep_time_ns": prep_time_ns,
    "epsilon_ramp_ns": epsilon_ramp_ns,
    "gate_time_us": gate_time_us,
    "gate_end_DQD1_spin_basis_up": float(p_up_gate[-1]),
    "gate_end_DQD1_spin_basis_down": float(p_down_gate[-1]),
    "gate_end_charge_right_1": float(charge_right_1[-1]),
    "gate_end_charge_right_2": float(charge_right_2[-1]),
    "readout_DQD1_spin_basis_up": float(p_up_readout[-1]),
    "readout_DQD1_spin_basis_down": float(p_down_readout[-1]),
    "readout_charge_right_1": float(readout_charge_right_1[-1]),
    "readout_charge_right_2": float(readout_charge_right_2[-1]),
}
readout_summary


## 5. Bell-state analysis with the optimized workflow prep

This section uses the optimized prep and gate-hold timings from `two_qubit_bell_state_optimization.py`. It keeps the same finite-detuning prep model, ramps to `epsilon = 0`, holds at the optimized resonator-mediated gate point, and ramps back for readout. The Bell diagnostics are evaluated in the diagonalized spin basis, including the `R,R`-conditioned spin state and the phase-optimized odd Bell state.


In [ ]:
bell_target = np.array([0.0, 1.0, 1.0j, 0.0], dtype=complex) / np.sqrt(2.0)
optimized_bell_prep_time_ns = 4.951
optimized_bell_gate_hold_us = 101.450

optimized_prep_idx = int(np.argmin(np.abs(prep_scan_ns - optimized_bell_prep_time_ns)))
optimized_bell_prep_time_ns = float(prep_scan_ns[optimized_prep_idx])

bell_gate_window_us = 1.0
gate_bell_tlist_us = np.linspace(
    max(0.0, optimized_bell_gate_hold_us - bell_gate_window_us),
    optimized_bell_gate_hold_us + bell_gate_window_us,
    401,
)
optimized_gate_idx = int(np.argmin(np.abs(gate_bell_tlist_us - optimized_bell_gate_hold_us)))
optimized_bell_gate_hold_us = float(gate_bell_tlist_us[optimized_gate_idx])

best_prep_state = U_full_spin.dag() * res_prep.states[optimized_prep_idx]
best_ramp_tlist_us, best_ramp_states = ramp_detuning_to_zero(
    dqd,
    best_prep_state,
    epsilon_prepare_uev,
    epsilon_ramp_us,
    ramp_samples,
    prep_opts,
)
psi_after_best_ramp = best_ramp_states[-1]
best_bell_states = evolve_under_constant_hamiltonian(dqd.H_static, psi_after_best_ramp, gate_bell_tlist_us)

bell_metric_samples = [bell_metrics(state, dqd, bell_target) for state in best_bell_states]
projected_bell_curve = np.array([sample["projected_bell_overlap"] for sample in bell_metric_samples])
qubit_weight_curve = np.array([sample["qubit_subspace_weight"] for sample in bell_metric_samples])
full_bell_weight_curve = np.array([sample["full_state_bell_weight"] for sample in bell_metric_samples])
gate_spin_phase_curve = np.array([sample["spin_phase_optimized_bell_overlap"] for sample in bell_metric_samples])
gate_rr_phase_curve = np.array([sample["rr_spin_phase_optimized_bell_overlap"] for sample in bell_metric_samples])
gate_rr_weighted_phase_curve = np.array([
    sample["rr_spin_weighted_phase_optimized_bell_overlap"] for sample in bell_metric_samples
])

best_gate_state = best_bell_states[optimized_gate_idx]
gate_metrics_best = bell_metric_samples[optimized_gate_idx]
best_readout_ramp_tlist_us, best_readout_ramp_states = ramp_detuning(
    dqd,
    best_gate_state,
    0.0,
    epsilon_prepare_uev,
    epsilon_ramp_us,
    ramp_samples,
    prep_opts,
)
best_readout_state = best_readout_ramp_states[-1]
readout_metrics_best = bell_metrics(best_readout_state, dqd, bell_target)

readout_spin_rho_best = spin_basis_two_spin_density_matrix(best_readout_state, dqd)
rr_spin_labels, rr_spin_populations, rr_spin_weight, readout_rr_spin_rho_best = spin_basis_rr_spin_populations(best_readout_state, dqd)
readout_rr_spin_phase = readout_metrics_best["rr_spin_phase_optimized_bell_phase_rad"]
phase_optimized_bell_target = odd_bell_target_with_phase(readout_rr_spin_phase)
phase_optimized_bell_ket = qt.Qobj(phase_optimized_bell_target[:, None], dims=[[2, 2], [1, 1]])

bell_summary = {
    "prep_time_ns": optimized_bell_prep_time_ns,
    "gate_hold_us": optimized_bell_gate_hold_us,
    "gate_spin_phase_optimized_bell_overlap": gate_metrics_best["spin_phase_optimized_bell_overlap"],
    "gate_spin_concurrence": gate_metrics_best["spin_concurrence"],
    "gate_RR_spin_weight": gate_metrics_best["rr_spin_weight"],
    "gate_RR_spin_phase_optimized_bell_overlap": gate_metrics_best["rr_spin_phase_optimized_bell_overlap"],
    "gate_RR_spin_weighted_phase_optimized_bell_overlap": gate_metrics_best[
        "rr_spin_weighted_phase_optimized_bell_overlap"
    ],
    "gate_RR_spin_concurrence": gate_metrics_best["rr_spin_concurrence"],
    "readout_spin_phase_optimized_bell_overlap": readout_metrics_best["spin_phase_optimized_bell_overlap"],
    "readout_spin_concurrence": readout_metrics_best["spin_concurrence"],
    "readout_RR_spin_weight": readout_metrics_best["rr_spin_weight"],
    "readout_RR_spin_bell_overlap_fixed_i": readout_metrics_best["rr_spin_bell_overlap"],
    "readout_RR_spin_phase_optimized_bell_overlap": readout_metrics_best[
        "rr_spin_phase_optimized_bell_overlap"
    ],
    "readout_RR_spin_weighted_phase_optimized_bell_overlap": readout_metrics_best[
        "rr_spin_weighted_phase_optimized_bell_overlap"
    ],
    "readout_RR_spin_concurrence": readout_metrics_best["rr_spin_concurrence"],
    "readout_RR_spin_phase_optimized_bell_phase_rad": readout_rr_spin_phase,
    "projected_bell_overlap_fixed_i_at_gate": gate_metrics_best["projected_bell_overlap"],
    "qubit_subspace_weight_at_gate": gate_metrics_best["qubit_subspace_weight"],
}

bell_prep_states = transform_states_from_spin_basis(res_prep.states[: optimized_prep_idx + 1], dqd)
bell_prep_times_ns = prep_scan_ns[: optimized_prep_idx + 1]
bell_ramp_states = best_ramp_states[1:]
bell_ramp_times_ns = optimized_bell_prep_time_ns + 1_000.0 * best_ramp_tlist_us[1:]
bell_gate_sequence_states = best_bell_states[1 : optimized_gate_idx + 1]
bell_gate_times_ns = optimized_bell_prep_time_ns + epsilon_ramp_ns + 1_000.0 * gate_bell_tlist_us[1 : optimized_gate_idx + 1]
bell_readout_states = best_readout_ramp_states[1:]
bell_readout_times_ns = optimized_bell_prep_time_ns + epsilon_ramp_ns + 1_000.0 * optimized_bell_gate_hold_us + 1_000.0 * best_readout_ramp_tlist_us[1:]
bell_full_states = bell_prep_states + bell_ramp_states + bell_gate_sequence_states + bell_readout_states
bell_full_times_ns = np.concatenate([bell_prep_times_ns, bell_ramp_times_ns, bell_gate_times_ns, bell_readout_times_ns])
bell_basis_labels, bell_basis_populations = vacuum_spin_charge_basis_populations(bell_full_states, dqd)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(gate_bell_tlist_us, gate_spin_phase_curve, label="spin phase-optimized Bell")
axes[0].plot(gate_bell_tlist_us, gate_rr_phase_curve, label="RR phase-optimized Bell")
axes[0].plot(gate_bell_tlist_us, gate_rr_weighted_phase_curve, label="RR weighted phase Bell")
axes[0].plot(gate_bell_tlist_us, full_bell_weight_curve, label="fixed-i projected Bell weight", alpha=0.6)
axes[0].axvline(optimized_bell_gate_hold_us, color="black", linestyle="--", linewidth=1.0)
axes[0].set_xlabel("gate hold (us)")
axes[0].set_ylabel("value")
axes[0].set_title("Bell-state metrics near the optimized hold")
axes[0].legend()

axes[1].bar(["gg", "ge", "eg", "ee"], gate_metrics_best["projected_populations"])
axes[1].set_ylim(0.0, 1.0)
axes[1].set_ylabel("projected population")
axes[1].set_title("Fixed-i projected qubit populations at selected hold")

plt.tight_layout()

fig, axes = plt.subplots(4, 4, figsize=(14, 10), sharex=True, sharey=True)
for idx, ax in enumerate(axes.ravel()):
    ax.plot(bell_full_times_ns, bell_basis_populations[:, idx], linewidth=1.5)
    ax.axvline(optimized_bell_prep_time_ns, color="black", linestyle="--", linewidth=1.0)
    ax.axvline(optimized_bell_prep_time_ns + epsilon_ramp_ns, color="gray", linestyle="--", linewidth=1.0)
    ax.axvline(optimized_bell_prep_time_ns + epsilon_ramp_ns + 1_000.0 * optimized_bell_gate_hold_us, color="firebrick", linestyle="--", linewidth=1.0)
    ax.set_title(bell_basis_labels[idx])
    ax.set_ylim(0.0, 1.0)
for ax in axes[-1, :]:
    ax.set_xlabel("sequence time (ns)")
for ax in axes[:, 0]:
    ax.set_ylabel("population")
fig.suptitle("All 16 spin-charge-basis populations during Bell-state preparation", y=1.02)
plt.tight_layout()

prep_boundary_populations = bell_basis_populations[len(bell_prep_states) - 1]
plot_population_basis_bar(
    bell_basis_labels,
    prep_boundary_populations,
    "16 spin-charge-basis populations right after the optimized single-qubit prep",
)

ramp_boundary_populations = bell_basis_populations[len(bell_prep_states) + len(bell_ramp_states) - 1]
plot_population_basis_bar(
    bell_basis_labels,
    ramp_boundary_populations,
    "16 spin-charge-basis populations after the ramp to zero detuning",
)

readout_boundary_populations = bell_basis_populations[-1]
plot_population_basis_bar(
    bell_basis_labels,
    readout_boundary_populations,
    "16 spin-charge-basis populations after the readout ramp back to finite detuning",
)

spin_rho_best = readout_spin_rho_best
spin_basis_labels = ["++", "+-", "-+", "--"]
spin_basis_kets = [
    tensor(basis(2, 0), basis(2, 0)),
    tensor(basis(2, 0), basis(2, 1)),
    tensor(basis(2, 1), basis(2, 0)),
    tensor(basis(2, 1), basis(2, 1)),
]
spin_basis_populations = [float(qt.expect(ket * ket.dag(), spin_rho_best).real) for ket in spin_basis_kets]
ideal_spin_rho = qt.Qobj(np.outer(phase_optimized_bell_target, phase_optimized_bell_target.conj()), dims=[[2, 2], [2, 2]])

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes[0, 0].bar(spin_basis_labels, spin_basis_populations)
axes[0, 0].set_ylim(0.0, 1.0)
axes[0, 0].set_ylabel("population")
axes[0, 0].set_title("Charge-traced spin populations")

im1 = axes[0, 1].imshow(np.abs(spin_rho_best.full()), vmin=0.0, vmax=1.0, cmap="viridis")
axes[0, 1].set_xticks(range(4), spin_basis_labels)
axes[0, 1].set_yticks(range(4), spin_basis_labels)
axes[0, 1].set_title(r"$|\rho_{spin}|$ after tracing charge")
fig.colorbar(im1, ax=axes[0, 1], fraction=0.046, pad=0.04)

im2 = axes[0, 2].imshow(np.abs(ideal_spin_rho.full()), vmin=0.0, vmax=1.0, cmap="viridis")
axes[0, 2].set_xticks(range(4), spin_basis_labels)
axes[0, 2].set_yticks(range(4), spin_basis_labels)
axes[0, 2].set_title(r"$|\rho_{Bell}(\phi)|$ target")
fig.colorbar(im2, ax=axes[0, 2], fraction=0.046, pad=0.04)

axes[1, 0].bar(rr_spin_labels, rr_spin_populations)
axes[1, 0].set_ylim(0.0, 1.0)
axes[1, 0].set_ylabel("conditional population")
axes[1, 0].set_title(f"RR-conditioned spin populations, weight={rr_spin_weight:.4g}")

im3 = axes[1, 1].imshow(np.abs(readout_rr_spin_rho_best.full()), vmin=0.0, vmax=1.0, cmap="viridis")
axes[1, 1].set_xticks(range(4), rr_spin_labels)
axes[1, 1].set_yticks(range(4), rr_spin_labels)
axes[1, 1].set_title(r"$|\rho_{spin}|$ conditioned on charge $R,R$")
fig.colorbar(im3, ax=axes[1, 1], fraction=0.046, pad=0.04)

im4 = axes[1, 2].imshow(np.abs(ideal_spin_rho.full()), vmin=0.0, vmax=1.0, cmap="viridis")
axes[1, 2].set_xticks(range(4), rr_spin_labels)
axes[1, 2].set_yticks(range(4), rr_spin_labels)
axes[1, 2].set_title(r"$|\rho_{Bell}(\phi)|$ target")
fig.colorbar(im4, ax=axes[1, 2], fraction=0.046, pad=0.04)

plt.tight_layout()

bell_summary


## Follow-up items

The notebook is still symmetric because that is the current requested workflow. The next extension is to lift that symmetry and allow independent DQD1 and DQD2 parameter sets, including different preparation frequencies and different readout projections if needed.

A published workspace copy of this notebook is written to `python/notebooks/published/two_qubit_gate_with_resonator_published.ipynb`.
